# Train the yard command adapter (LoRA)

Adapted from the LocoLabo capstone notebook by Michael Borck (Curtin, Semester 2 2026), for the yard voice-command practice project.

It takes `data/train_conversations.jsonl` and trains a LoRA adapter on **Qwen3-1.7B**, then exports a file Ollama can run.

## Before you start

- Run `python src/generate_data.py` on your laptop so `data/train_conversations.jsonl` exists (about 210 examples).
- Never train on `data/test.jsonl`. It is kept for testing.
- **Runtime → Change runtime type → T4 GPU.** The free tier is enough.

With this little data, training takes minutes. Installing, downloading the model and exporting take most of the time (about 30 to 60 minutes in total).


## Step 0: Confirm you have a GPU

If this errors or shows no GPU, fix the runtime type before going further.


In [ ]:
!nvidia-smi


## Step 1: Install Unsloth

Unsloth makes LoRA training fit in a free Colab GPU. This takes a few minutes and
may ask you to restart the session. If it does, restart and run from here.


In [ ]:
!pip install -q unsloth unsloth_zoo


## Step 2: Settings

Everything you might want to change lives here.

The defaults are sensible starting points and match the other adapters in the
project, so your results stay comparable. Change them deliberately, one at a
time, and record what you changed and why in `TRAINING_LOG.md`.

| Setting | What it does |
|---|---|
| `LORA_R` | How much capacity the specialisation layer has. Higher learns more, overfits sooner. 8 to 32 is the useful range. |
| `EPOCHS` | How many times the model sees your whole dataset. Too many and it memorises rather than generalises. |
| `LEARNING_RATE` | How big a step each update takes. Lower it if the loss jumps around. |
| `MAX_SEQ_LENGTH` | Longest example, in tokens. Raise it if your answers are long, at the cost of memory. |


In [ ]:
ADAPTER_NAME = "yard"   # lowercase, no spaces

BASE_MODEL = "unsloth/Qwen3-1.7B-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 512

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

LEARNING_RATE = 2e-4
EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 4          # effective batch size is BATCH_SIZE * GRAD_ACCUM

QUANT_METHOD = "q4_k_m"
SEED = 42               # leave this alone so your runs are reproducible


## Step 3: Load your training data

Run the cell and pick `train_conversations.jsonl` from the `data` folder on your laptop.


In [ ]:
from google.colab import files

uploaded = files.upload()
DATA_PATH = next(iter(uploaded))
print(f"Using {DATA_PATH}")


## Step 4: Check the data before training on it

Worth its few seconds. A format error caught here costs nothing; the same error
caught after a three hour training run costs an afternoon.

This checks every line parses, has the `conversations` structure the trainer
expects, alternates user then assistant, and has no empty answers. It also flags
duplicates, which quietly waste training capacity.


In [ ]:
import json
from collections import Counter

records, problems = [], []
with open(DATA_PATH) as f:
    for n, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
        except json.JSONDecodeError as e:
            problems.append(f"line {n}: not valid JSON ({e})")
            continue

        convo = rec.get("conversations")
        if not isinstance(convo, list) or len(convo) < 2:
            problems.append(f"line {n}: needs a 'conversations' list of at least 2 turns")
            continue
        roles = [t.get("role") for t in convo]
        if roles[0] != "user" or roles[1] != "assistant":
            problems.append(f"line {n}: first two turns must be user then assistant, got {roles[:2]}")
            continue
        if not str(convo[1].get("content", "")).strip():
            problems.append(f"line {n}: assistant answer is empty")
            continue
        records.append(rec)

questions = [r["conversations"][0].get("content", "") for r in records]
dupes = [q for q, c in Counter(questions).items() if c > 1]

print(f"Valid examples: {len(records)}")
print(f"Duplicate questions: {len(dupes)}")
if problems:
    print(f"\nProblems found: {len(problems)}")
    for p in problems[:10]:
        print("  " + p)
    if len(problems) > 10:
        print(f"  ... and {len(problems) - 10} more")

if len(records) < 150:
    print(f"\nNote: only {len(records)} examples. Run generate_data.py with a bigger --n.")

assert records, "No usable examples. Fix the format before continuing."


### Look at your data

Read a couple of examples with your own eyes. The model learns to imitate the
assistant answers, so if these are not good, nothing downstream will be.


In [ ]:
for rec in records[:2]:
    turns = rec["conversations"]
    print("USER:     ", turns[0]["content"][:300])
    print("ASSISTANT:", turns[1]["content"][:300])
    print("-" * 70)


## Step 5: Load the base model

This is the model you are trying to beat. It loads in 4-bit precision, which is
what makes a 1.7 billion parameter model fit in a free Colab GPU.

First run downloads several GB, so it takes a few minutes.


In [ ]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    full_finetuning=False,
)


## Step 6: Add the specialisation layer

This is the LoRA part, and the heart of the whole approach.

Rather than retraining 1.7 billion parameters, which no free GPU could manage,
you freeze the base model and train a small set of extra weights alongside it.
The printout below shows how small: typically well under 1% of the total.

That is the claim this project is testing. A tiny number of well-chosen trained
parameters, on the right data, beating a much larger general model at one task.


In [ ]:
model = FastModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100 * trainable / total:.3f}%)")


## Step 7: Format the examples

Every model expects conversations laid out in its own particular way, with its
own markers for where a turn starts and ends. `apply_chat_template` handles that
so you do not have to.

The printed sample shows what the model actually sees. Worth a look once.


In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(records)

def format_conversations(examples):
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in examples["conversations"]
    ]
    return {"text": texts}

dataset = dataset.map(format_conversations, batched=True)
print(dataset[0]["text"][:600])


## Step 8: Train

**Watch the loss.** It should fall and then flatten out. If it falls to almost nothing, the model is memorising the examples rather than learning the pattern; try fewer epochs. If it bounces around without settling, lower the learning rate.

Write the final loss and the settings you used in your notes, even if the run fails.


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=TrainingArguments(
        output_dir="checkpoints",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        fp16=True,
        logging_steps=5,
        save_strategy="epoch",
        seed=SEED,
        optim="adamw_8bit",
        report_to="none",
    ),
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
)

stats = trainer.train()
print(stats)


## Step 9: Export for Ollama

This merges your trained layer into the base model and compresses the result to
about 1 to 1.5 GB, which is the file Ollama runs.

Takes 10 to 20 minutes and is disk hungry. If Colab runs out of space, restart
the runtime and run everything except this cell again.


In [ ]:
OUTPUT_DIR = f"{ADAPTER_NAME}_gguf"

model.save_pretrained_gguf(OUTPUT_DIR, tokenizer, quantization_method=QUANT_METHOD)

import os
for f in os.listdir(OUTPUT_DIR):
    if f.endswith(".gguf"):
        size_gb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / (1024 ** 3)
        print(f"{f}  ({size_gb:.1f} GB)")


## Step 10: Get the file out of Colab

Copy it to Google Drive rather than downloading through the browser. **Do this before closing the tab**: Colab deletes everything when the session ends.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p "/content/drive/MyDrive/yard-voice-commands/{ADAPTER_NAME}"
!cp {OUTPUT_DIR}/*.gguf "/content/drive/MyDrive/yard-voice-commands/{ADAPTER_NAME}/"
!ls -lh "/content/drive/MyDrive/yard-voice-commands/{ADAPTER_NAME}/"


## What happens next (on your laptop)

1. Download the `.gguf` file from Google Drive into a folder, e.g. `models/` (keep it out of GitHub; it's too big).
2. Make a file called `Modelfile` next to it with one line (use your file's real name):

```
FROM ./yard.Q4_K_M.gguf
```

3. Register it with Ollama:

```
ollama create yard-lora -f Modelfile
ollama run yard-lora
```

4. Test it on `data/test.jsonl` with the **same prompt** the training used (`src/prompt.py`) and compare it with the base model `qwen3:1.7b` on the same test set.

If the score did not improve, that's a real result too. Read some of the wrong answers before concluding anything: check whether the labels are right, whether the base model was already good, and whether you trained too little or too much.
